# 01. Quickstart

The canonical OTTER π, a single-region query, and the top-K human partners of a mouse region with their multi-source trust tier.

This notebook does not re-fit the model. It loads pre-computed outputs from `outputs/coupling/`. To regenerate them, run `experiments/anchor_packs/compose_all.py` and then this notebook again.

**Sections:**
1. Setup, load π, atlases, trust map
2. Single-region query (interactive)
3. Compare two π files (canonical vs the pre-warp coupling it superseded)
4. Bulk region translation
5. 3D brain visualisation

In [ ]:
# Setup
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from otter.data import load_cached, load_pi

ROOT = Path.cwd().parent
ANN = ROOT / 'outputs' / 'anndata'
COUP = ROOT / 'outputs' / 'coupling'
LOGS = ROOT / 'outputs' / 'logs'

# Load atlases (instant, h5ad caches)
M, _ = load_cached('mouse', cache_dir=ANN)
H, _ = load_cached('human', cache_dir=ANN)
print(f'Mouse parcels: {len(M.var)}, Human parcels: {len(H.var)}')

# The canonical coupling. load_pi() defaults to pi_canonical.npy. Always prefer load_pi()
# over a bare np.load of a filename: it resolves the repo root and fetches the data bundle
# if it is missing. 02_methodology.ipynb section 12 documents how it was built and selected.
pi_canon = load_pi()

dep = json.loads((LOGS / 'section5_canonical_sweep.json').read_text())['deploy']
print(f"π canonical  {pi_canon.shape}  (cell {dep['cell']}: "
      f"xyz_weight={dep['xyz_weight']}, epsilon={dep['epsilon']})")

# Multi-source trust map computed on the canonical π
trust = np.load(COUP / 'trust_multisource_canonical.npz', allow_pickle=True)
trust_tier = trust['evidence_tier']
print(f'\nTrust tiers (over {len(trust_tier)} mouse parcels):')
for t in ['anchored_and_validated', 'anchored_only', 'validated_only', 'structural', 'low_evidence']:
    n = int((trust_tier == t).sum())
    print(f'  {t:25s} {n:4d}  ({n/len(trust_tier):>6.1%})')


## 2. Interactive single-region query

The dropdown selects a mouse region and the slider sets top-K. The display shows:
- the top-K human partners with their MNI coords and region names
- the predicted region's trust tier (multi-source evidence label)
- the row's mass concentration (sharpness of the prediction)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Build a sorted list of unique mouse regions (most-represented first)
region_counts = M.var['region'].value_counts()
mouse_regions = region_counts.index.tolist()

# Widgets
region_dropdown = widgets.Dropdown(
    options=mouse_regions, value=mouse_regions[0],
    description='Mouse region:', layout=widgets.Layout(width='500px'),
    style={'description_width': '120px'},
)
parcel_dropdown = widgets.Dropdown(
    options=[], description='Mouse parcel:',
    layout=widgets.Layout(width='500px'),
    style={'description_width': '120px'},
)
topk_slider = widgets.IntSlider(value=5, min=1, max=10, step=1,
                                  description='top-K:',
                                  style={'description_width': '120px'})
out = widgets.Output()

def update_parcels(*_):
    region = region_dropdown.value
    parcels = M.var[M.var['region'] == region].index.tolist()
    parcel_dropdown.options = parcels
    parcel_dropdown.value = parcels[0] if parcels else None

def render(*_):
    with out:
        clear_output()
        parcel_id = parcel_dropdown.value
        if parcel_id is None:
            print('No parcel selected.')
            return
        pi = pi_canon
        # Find the row index for this parcel
        row = M.var.index.get_loc(parcel_id)
        k = topk_slider.value
        topk = pi[row].argsort()[::-1][:k]
        scores = pi[row][topk]

        tier = trust_tier[row]
        concentration = pi[row].max() / pi[row].sum()

        print(f'Mouse parcel: {parcel_id} ({M.var.iloc[row]["region"]})')
        print(f'  xyz: ({M.var.iloc[row].x:+.2f}, {M.var.iloc[row].y:+.2f}, {M.var.iloc[row].z:+.2f}) mm')
        print(f'  Trust tier:        {tier}')
        print(f'  Row concentration: {concentration:.1%} (peak / row sum)')
        print(f'\nTop {k} human partners:')
        rows = []
        for r in topk:
            rows.append({
                'human_parcel': H.var.index[r],
                'region': H.var.iloc[r]['region'],
                'x':  f'{H.var.iloc[r].x:+.1f}',
                'y':  f'{H.var.iloc[r].y:+.1f}',
                'z':  f'{H.var.iloc[r].z:+.1f}',
                'pi':       f'{pi[row, r]:.4f}',
                'pi_frac':  f'{pi[row, r] / pi[row].sum():.1%}',
            })
        display(pd.DataFrame(rows))

region_dropdown.observe(update_parcels, names='value')
update_parcels()
for w in (region_dropdown, parcel_dropdown, topk_slider):
    w.observe(render, names='value')

display(widgets.VBox([region_dropdown, parcel_dropdown, topk_slider, out]))
render()

## 3. The canonical π against the pre-warp coupling

The cell below shows, for the parcel selected above, how the canonical coupling (anchor-warped
spatial cost + region packs) and the pre-warp coupling (packs only) differ. That makes the effect
of the anchor-driven warp visible per parcel.


In [ ]:
def top_partners(parcel_id, k=5):
    """Top-k human partners for a mouse parcel, as a share of that parcel's routed mass.

    The canonical coupling is soft (see 02_methodology.ipynb section 12), so raw entries of pi
    are small. Reporting each partner's SHARE of the row gives the interpretable
    quantity, which is what the coupling asserts about where that parcel maps.
    """
    row = M.var.index.get_loc(parcel_id)
    share = pi_canon[row] / pi_canon[row].sum()
    top = share.argsort()[::-1][:k]
    return pd.DataFrame([{
        'rank': i + 1,
        'human region': H.var.iloc[r]['region'][:40],
        'share of routed mass': f'{share[r]:.1%}',
    } for i, r in enumerate(top)])


motor_parcel = M.var[M.var['region'].str.contains('Motor', case=False, na=False)].index[0]
print(f'Mouse parcel {motor_parcel} ({M.var.loc[motor_parcel, "region"]}):')
top_partners(motor_parcel, k=5)

## 4. Bulk region translation

π is aggregated across all member parcels of a mouse region, and the top human regions are listed.

In [ ]:
def translate_region(region_query, pi=pi_canon, top_k=5):
    """Aggregate π over mouse parcels matching region_query, return top-K human regions."""
    mask = M.var['region'].str.contains(region_query, case=False, na=False)
    if mask.sum() == 0:
        return f'No mouse parcels match {region_query!r}'
    pi_M = pi[mask].sum(axis=0)
    pi_M /= pi_M.sum()
    # Aggregate by human region name
    h_regions = H.var['region'].copy()
    agg = pd.Series(pi_M).groupby(h_regions.values).sum().sort_values(ascending=False).head(top_k)
    out = pd.DataFrame({
        'human_region': agg.index,
        'mass_share':   [f'{v:.1%}' for v in agg.values],
    })
    print(f'Mouse "{region_query}", {int(mask.sum())} parcels, top-{top_k} human partners:')
    return out

# NB: this mouse atlas has 42 named regions; the rest are numbered parcels.
translate_region('Somatosensory', top_k=10)

Other regions.

In [ ]:
# Visualise where Motor maps to
print(translate_region('Motor', top_k=8))
print('---')
print(translate_region('Amygdala', top_k=8))
print('---')
print(translate_region('Thalamus', top_k=8))

## 5. 3D brain view

The mouse brain coloured by Garin functional network. Each dot is a parcel.

In [ ]:
from otter.viz.notebook import plot_brain_3d
plot_brain_3d(M, color_by='network', title='Mouse atlas, network coloring')

## Further reading

- `03_coupling.ipynb`, the per-parcel evidence tiers and what they grade.
- `04_cost_terms_and_supervision.ipynb`, the contribution of the anchors at parcel level.
- `02_methodology.ipynb`, the FGW solver step by step; section 12 there covers the selection of
  the canonical π's hyperparameters.
- `docs/04_anchor_packs.md`, adding new packs.

The snippet in `README.md` covers programmatic use.

## The coupling and a worked query

In [ ]:
# Query the coupling. The coupling analyses are reproduced in notebooks/03_coupling.ipynb;
# this cell demonstrates the query.
import numpy as np

# pi_canon is the canonical coupling loaded in the setup cell. The notebook also offers a
# `pi` handle switched by a dropdown; use the explicit name so this cell does not depend
# on widget state when the notebook is run non-interactively.
P = pi_canon / pi_canon.sum(1, keepdims=True).clip(1e-12)
top_p = P.max(axis=1)
print(f"coupling {pi_canon.shape[0]:,} mouse x {pi_canon.shape[1]:,} human parcels")
print(f"median top-target probability {np.median(top_p):.2f}; "
      f"> 0.5 for {(top_p > 0.5).mean() * 100:.0f} % of parcels")
print("\nThe coupling is a distribution rather than a lookup table. Most mouse parcels spread")
print("their mass over several human parcels, and that spread is calibrated.")
print("\nThe coupling analyses are reproduced in notebooks/03_coupling.ipynb")

In [ ]:
# Topographic preservation. Two mouse parcels that lie close together should route to human
# centroids that lie close together, which separates a property of the coupling from a solver
# artefact.
import numpy as np
from scipy.stats import pearsonr

human_xyz = H.var[['x', 'y', 'z']].to_numpy(float)
mouse_xyz = M.var[['x', 'y', 'z']].to_numpy(float)

def routed_centroids(p):
    return (p / np.maximum(p.sum(axis=1, keepdims=True), 1e-12)) @ human_xyz

def pdist_flat(X):
    D = np.linalg.norm(X[:, None, :] - X[None, :, :], axis=-1)
    return D[np.triu_indices(len(X), k=1)]

rng = np.random.default_rng(0)
sub = rng.choice(pi_canon.shape[0], size=600, replace=False)
dm = pdist_flat(mouse_xyz[sub])
r_obs = float(pearsonr(dm, pdist_flat(routed_centroids(pi_canon)[sub]))[0])
r_null = float(pearsonr(dm, pdist_flat(
    routed_centroids(pi_canon[rng.permutation(pi_canon.shape[0])])[sub]))[0])

print(f"mouse distance vs routed human distance:  r = {r_obs:.2f}")
print(f"permuted-coupling null:                   r = {r_null:+.2f}")
print("\nRouting preserves topography, so the orderly diagonal is a property of the coupling")
print("rather than of the solver. See notebooks/03_coupling.ipynb")